# 🧩 Mini-Lab: System Prompt Behaviors

**Module 4: Prompt Engineering & Security** | **Duration: ~20 min** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** the distinct roles of system and user messages in the Chat Completions API
2. **Design** system prompts that shape an LLM's tone, style, and constraints
3. **Apply** role prompting to make the model adopt specific expert personas
4. **Compare** how different system prompts change the model's response to the same user message

## Target Concepts

| Concept | Description |
|---------|-------------|
| Prompt Engineering | The discipline of designing, refining, and optimizing inputs to language models to reliably produce desired outputs |
| Prompt Design | Principles of structuring effective prompts for LLM applications |
| System Prompt | The system-level message that sets the AI's behavior, personality, and constraints |
| User Prompt | The user-level message that contains the actual request or question |
| Role Prompting | Assigning a specific role or persona to the AI to control its behavior |

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()  # uses OPENAI_API_KEY from .env

MODEL = "gpt-4o-mini"

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def chat(system, user):
    """Send a system + user message and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

## 1. System vs. User Messages

The Chat Completions API uses a **messages** array where each message has a **role**:

| Role | Purpose |
|------|--------|
| `system` | Sets the AI's behavior, personality, and constraints. The model treats this as high-level instructions. |
| `user` | Contains the actual question or request from the end user. |
| `assistant` | The model's previous responses (used in multi-turn conversations). |

The **system prompt** is your primary tool for controlling *how* the model responds, while the **user prompt** controls *what* it responds to.

Let's see a basic example with no system prompt versus one with a system prompt.

In [ ]:
user_question = "What is photosynthesis?"

# Without a meaningful system prompt
response_default = chat("You are a helpful assistant.", user_question)

md("### Default System Prompt")
md(response_default)

In [ ]:
# With a specific system prompt
response_custom = chat(
    "You are a science teacher for 5-year-olds. "
    "Explain everything using simple words and fun analogies. "
    "Keep answers to 2-3 sentences.",
    user_question
)

md("### Custom System Prompt (Teacher for 5-year-olds)")
md(response_custom)

Notice how the **same user question** produces very different responses depending on the system prompt. The system prompt controlled the tone, complexity, and length.

## 2. Anatomy of a Good System Prompt

An effective system prompt typically includes some combination of:

1. **Identity** — Who is the AI? (e.g., "You are a senior Python developer.")
2. **Behavior** — How should it behave? (e.g., "Be concise and direct.")
3. **Constraints** — What should it avoid? (e.g., "Never provide medical advice.")
4. **Output format** — How should responses be structured? (e.g., "Always respond in bullet points.")

Let's see how adding these elements shapes the output.

In [ ]:
system_prompt = """You are a senior software engineer at a tech company.

Behavior:
- Be concise and technical
- Provide code examples when relevant
- Mention trade-offs when suggesting solutions

Constraints:
- Do not suggest deprecated libraries
- Keep responses under 150 words

Output format:
- Use markdown with headers and code blocks"""

response = chat(system_prompt, "How should I handle errors in a REST API?")

md(response)

## 3. Role Prompting — Changing Personas

**Role prompting** means assigning the model a specific expert persona. The same question asked to different "roles" produces answers tailored to that perspective.

Let's ask three different roles the same question and compare.

In [ ]:
user_question = "How can I improve my company's customer retention?"

roles = {
    "Marketing Strategist": (
        "You are a marketing strategist with 15 years of experience. "
        "Focus on branding, engagement campaigns, and customer loyalty programs. "
        "Keep your answer to 3 bullet points."
    ),
    "Data Scientist": (
        "You are a data scientist specializing in customer analytics. "
        "Focus on data-driven approaches, metrics, and predictive modeling. "
        "Keep your answer to 3 bullet points."
    ),
    "Customer Support Lead": (
        "You are a customer support team lead with deep empathy for users. "
        "Focus on service quality, feedback loops, and human connection. "
        "Keep your answer to 3 bullet points."
    )
}

for role_name, system in roles.items():
    response = chat(system, user_question)
    md(f"### 🎭 {role_name}")
    md(response)
    md("---")

Each role brought a **different perspective** to the exact same question. This is the power of role prompting — you steer the model's expertise and framing without changing the user's question.

## 4. System Prompt Constraints in Action

System prompts can enforce **constraints** — rules the model should follow. Let's see how a constraint changes behavior.

In [ ]:
# Without constraint
response_no_constraint = chat(
    "You are a helpful cooking assistant.",
    "Give me a recipe for pasta."
)

md("### Without Constraint")
md(response_no_constraint)

In [ ]:
# With constraints
response_constrained = chat(
    "You are a helpful cooking assistant.\n\n"
    "Constraints:\n"
    "- Only suggest vegan recipes\n"
    "- Always include estimated prep time\n"
    "- List exactly 5 ingredients or fewer",
    "Give me a recipe for pasta."
)

md("### With Constraints (vegan, prep time, ≤5 ingredients)")
md(response_constrained)

The constraints in the system prompt shaped the response to follow specific rules — vegan only, with a time estimate, and a limited ingredient list. This is how you build reliable, consistent AI behaviors in applications.

## 🎯 Summary

### Key Takeaways

1. **System vs. User messages** — the system message controls *how* the model behaves; the user message controls *what* it responds to
2. **System prompt anatomy** — effective system prompts combine identity, behavior rules, constraints, and output format instructions
3. **Role prompting** — assigning a persona (e.g., "You are a data scientist") steers the model's expertise and perspective without changing the question
4. **Constraints** — adding explicit rules in the system prompt enforces consistent, predictable outputs for real applications